# Eksperimen Machine Learning - Wine Quality Dataset

**Nama:** ardir  
**Kelas:** Membangun Sistem Machine Learning  
**Dataset:** Wine Quality (UCI ML Repository)  
**Tanggal:** 2026-05-18

---

## Daftar Isi
1. Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Modeling (Eksperimen Awal)
5. Evaluasi Model
6. Kesimpulan

## 1. Data Loading

Pada tahap ini, kita akan memuat dataset Wine Quality dari UCI ML Repository. Dataset ini terdiri dari dua subset: Red Wine dan White Wine.

In [ ]:
# Import library yang diperlukan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import warnings
warnings.filterwarnings("ignore")

print("Library berhasil dimuat!")

In [ ]:
# Load dataset dari UCI ML Repository
RED_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
WHITE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

df_red = pd.read_csv(RED_URL, sep=";")
df_white = pd.read_csv(WHITE_URL, sep=";")

# Tambahkan kolom wine_type
df_red["wine_type"] = "red"
df_white["wine_type"] = "white"

# Gabungkan dataset
df = pd.concat([df_red, df_white], axis=0, ignore_index=True)

print(f"Red Wine: {len(df_red)} baris")
print(f"White Wine: {len(df_white)} baris")
print(f"Total Dataset: {len(df)} baris, {df.shape[1]} kolom")

In [ ]:
# Tampilkan 5 baris pertama
df.head()

In [ ]:
# Informasi dataset
df.info()

In [ ]:
# Statistik deskriptif
df.describe()

## 2. Exploratory Data Analysis (EDA)

Pada tahap ini kita akan mengeksplorasi distribusi data, korelasi antar fitur, dan keseimbangan kelas.

In [ ]:
# Cek missing values
print("Missing Values:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Cek duplikasi
print(f"Jumlah baris duplikat: {df.duplicated().sum()}")

In [ ]:
# Distribusi target (quality)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot distribusi quality
quality_counts = df["quality"].value_counts().sort_index()
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(quality_counts)))
axes[0].bar(quality_counts.index, quality_counts.values, color=colors)
axes[0].set_xlabel("Quality Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribusi Wine Quality Score")

# Pie chart wine type
wine_type_counts = df["wine_type"].value_counts()
axes[1].pie(wine_type_counts, labels=wine_type_counts.index, autopct="%1.1f%%",
            colors=["#E74C3C", "#F4D03F"], startangle=90)
axes[1].set_title("Proporsi Red vs White Wine")

plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur numerik
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    if i < len(axes):
        axes[i].hist(df[col], bins=30, color="#3498DB", alpha=0.7, edgecolor="black")
        axes[i].set_title(col, fontsize=10)
        axes[i].set_ylabel("Frequency")

# Hapus subplot yang tidak terpakai
for j in range(len(num_cols), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Distribusi Fitur Numerik", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
corr_matrix = df[num_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, linewidths=0.5,
            square=True, cbar_kws={"shrink": 0.8})
plt.title("Korelasi Antar Fitur", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot per quality
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
important_features = ["alcohol", "volatile acidity", "sulphates", "citric acid", "density", "pH"]

for i, col in enumerate(important_features):
    sns.boxplot(data=df, x="quality", y=col, ax=axes[i], palette="viridis")
    axes[i].set_title(f"{col} vs Quality")

plt.suptitle("Fitur Penting vs Quality Score", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Perbandingan Red vs White Wine
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(important_features):
    sns.violinplot(data=df, x="wine_type", y=col, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} by Wine Type")

plt.suptitle("Perbandingan Red vs White Wine", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

Tahapan preprocessing meliputi:
- Handling missing values & duplicates
- Feature engineering
- Encoding fitur kategorikal
- Scaling fitur numerik
- Train-validation-test split

In [ ]:
# 3.1 Hapus duplikasi
print(f"Sebelum: {len(df)} baris")
df_clean = df.drop_duplicates()
print(f"Sesudah: {len(df_clean)} baris")
print(f"Duplikasi dihapus: {len(df) - len(df_clean)} baris")

In [ ]:
# 3.2 Feature Engineering
# Buat fitur baru
df_clean = df_clean.copy()
df_clean["total_acidity"] = df_clean["fixed acidity"] + df_clean["volatile acidity"]
df_clean["free_sulfur_ratio"] = df_clean["free sulfur dioxide"] / df_clean["total sulfur dioxide"].replace(0, 1)
df_clean["alcohol_density_ratio"] = df_clean["alcohol"] / df_clean["density"].replace(0, 1)

# Kategorikan quality
def categorize_quality(q):
    if q <= 4:
        return "low"
    elif q <= 6:
        return "medium"
    else:
        return "high"

df_clean["quality_category"] = df_clean["quality"].apply(categorize_quality)

print("Distribusi kelas:")
print(df_clean["quality_category"].value_counts())
print(f"\nTotal fitur: {df_clean.shape[1]}")

In [ ]:
# 3.3 Encoding
le_wine = LabelEncoder()
df_clean["wine_type"] = le_wine.fit_transform(df_clean["wine_type"])

le_quality = LabelEncoder()
df_clean["quality_label"] = le_quality.fit_transform(df_clean["quality_category"])

print("Label mapping:")
print(dict(zip(le_quality.classes_, le_quality.transform(le_quality.classes_))))

In [ ]:
# 3.4 Definisikan fitur dan target
drop_cols = ["quality_label", "quality", "quality_category"]
X = df_clean.drop(columns=drop_cols)
y = df_clean["quality_label"]

print(f"Fitur: {X.shape}")
print(f"Target: {y.shape}")
print(f"\nNama fitur: {X.columns.tolist()}")

In [ ]:
# 3.5 Train-Validation-Test Split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp
)

total = len(X)
print(f"Train: {X_train.shape[0]} samples ({X_train.shape[0]/total*100:.1f}%)")
print(f"Val:   {X_val.shape[0]} samples ({X_val.shape[0]/total*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} samples ({X_test.shape[0]/total*100:.1f}%)")

In [ ]:
# 3.6 Scaling
scaler = StandardScaler()
num_cols_scale = X_train.select_dtypes(include=[np.number]).columns.tolist()

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols_scale] = scaler.fit_transform(X_train[num_cols_scale])
X_val_scaled[num_cols_scale] = scaler.transform(X_val[num_cols_scale])
X_test_scaled[num_cols_scale] = scaler.transform(X_test[num_cols_scale])

print("Scaling selesai!")
print("\nContoh data setelah scaling:")
X_train_scaled.head()

## 4. Modeling (Eksperimen Awal)

Kita akan mencoba beberapa model untuk eksperimen awal:
1. Random Forest Classifier
2. Gradient Boosting Classifier
3. Logistic Regression

In [ ]:
# 4.1 Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)

rf_train_acc = accuracy_score(y_train, rf_model.predict(X_train_scaled))
rf_val_acc = accuracy_score(y_val, rf_model.predict(X_val_scaled))
rf_test_acc = accuracy_score(y_test, rf_model.predict(X_test_scaled))

print("Random Forest Results:")
print(f"  Train Accuracy: {rf_train_acc:.4f}")
print(f"  Val Accuracy:   {rf_val_acc:.4f}")
print(f"  Test Accuracy:  {rf_test_acc:.4f}")

In [ ]:
# 4.2 Gradient Boosting
gb_model = GradientBoostingClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
)
gb_model.fit(X_train_scaled, y_train)

gb_train_acc = accuracy_score(y_train, gb_model.predict(X_train_scaled))
gb_val_acc = accuracy_score(y_val, gb_model.predict(X_val_scaled))
gb_test_acc = accuracy_score(y_test, gb_model.predict(X_test_scaled))

print("Gradient Boosting Results:")
print(f"  Train Accuracy: {gb_train_acc:.4f}")
print(f"  Val Accuracy:   {gb_val_acc:.4f}")
print(f"  Test Accuracy:  {gb_test_acc:.4f}")

In [ ]:
# 4.3 Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

lr_train_acc = accuracy_score(y_train, lr_model.predict(X_train_scaled))
lr_val_acc = accuracy_score(y_val, lr_model.predict(X_val_scaled))
lr_test_acc = accuracy_score(y_test, lr_model.predict(X_test_scaled))

print("Logistic Regression Results:")
print(f"  Train Accuracy: {lr_train_acc:.4f}")
print(f"  Val Accuracy:   {lr_val_acc:.4f}")
print(f"  Test Accuracy:  {lr_test_acc:.4f}")

## 5. Evaluasi Model

Membandingkan performa semua model dan mengevaluasi model terbaik secara detail.

In [ ]:
# 5.1 Perbandingan Model
results = pd.DataFrame({
    "Model": ["Random Forest", "Gradient Boosting", "Logistic Regression"],
    "Train Acc": [rf_train_acc, gb_train_acc, lr_train_acc],
    "Val Acc": [rf_val_acc, gb_val_acc, lr_val_acc],
    "Test Acc": [rf_test_acc, gb_test_acc, lr_test_acc]
})

print("Perbandingan Performa Model:")
print(results.to_string(index=False))

# Plot perbandingan
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results))
width = 0.25

bars1 = ax.bar(x - width, results["Train Acc"], width, label="Train", color="#2196F3")
bars2 = ax.bar(x, results["Val Acc"], width, label="Validation", color="#4CAF50")
bars3 = ax.bar(x + width, results["Test Acc"], width, label="Test", color="#FF5722")

ax.set_ylabel("Accuracy")
ax.set_title("Perbandingan Performa Model", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(results["Model"])
ax.legend()
ax.set_ylim(0, 1.1)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.3f}", xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Classification Report model terbaik (Random Forest)
y_pred_test = rf_model.predict(X_test_scaled)
class_labels = ["high", "low", "medium"]

print("Classification Report - Random Forest (Test Set):")
print(classification_report(y_test, y_pred_test, target_names=class_labels))

In [ ]:
# 5.3 Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_labels, yticklabels=class_labels)
plt.title("Confusion Matrix - Random Forest", fontsize=14, fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# 5.4 Feature Importance
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(indices)))
plt.barh(range(len(indices)), importances[indices][::-1], color=colors)
plt.yticks(range(len(indices)), [X_train_scaled.columns[i] for i in indices[::-1]])
plt.xlabel("Importance")
plt.title("Feature Importances - Random Forest", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Kesimpulan

### Temuan Utama:
1. **Dataset**: Wine Quality dataset terdiri dari 6497 sampel (1599 red, 4898 white) dengan 12 fitur.
2. **EDA**: Distribusi quality cenderung ke kelas medium (5-6). Fitur `alcohol`, `volatile acidity`, dan `sulphates` memiliki korelasi kuat dengan quality.
3. **Preprocessing**: Dilakukan feature engineering (total_acidity, free_sulfur_ratio, alcohol_density_ratio), encoding, dan scaling.
4. **Model Terbaik**: Random Forest Classifier memberikan performa terbaik dibandingkan Gradient Boosting dan Logistic Regression.

### Langkah Selanjutnya:
- Implementasi automated preprocessing dengan `automate_ardir.py`
- Integrasi dengan MLflow untuk tracking eksperimen
- Hyperparameter tuning lebih lanjut
- Deployment model menggunakan FastAPI
- Monitoring dengan Prometheus dan Grafana